# 03 — Filtering, Sorting & Set Operations

**What's in this notebook:**
- `WHERE` — comparison operators, boolean logic, and precedence
- Pattern matching, ranges, and set membership — `LIKE` / `ILIKE`, `BETWEEN`, `IN`
- `NULL` in filters — the three-valued-logic recap and the `NOT IN` trap
- `ORDER BY` — direction, multi-column sorting, and `NULLS FIRST` / `NULLS LAST`
- `LIMIT`, `OFFSET`, and why deep pagination is slow
- Set operations — `UNION`, `UNION ALL`, `INTERSECT`, `EXCEPT`
- Common pitfalls to carry forward

This notebook uses the `customers` and `orders` tables from notebook 01, plus a small `prospects` table we add below for the set-operation examples.

In [ ]:
%load_ext sql
%sql duckdb:///learn.db

## 1. WHERE — operators, boolean logic, and precedence

`WHERE` runs **per row**. For each row from `FROM`, the engine evaluates the boolean expression; rows where it is `TRUE` survive, rows where it is `FALSE` or `UNKNOWN` are dropped.

**Comparison operators**: `=`, `<>` (or `!=`), `<`, `<=`, `>`, `>=`. Standard across dialects.

**Logical operators**: `AND`, `OR`, `NOT`. The trap is precedence — **`AND` binds tighter than `OR`**, the same as `*` binding tighter than `+`. When you mix them, parenthesize. The query you *think* you wrote and the query the parser sees can disagree without parentheses.

In [ ]:
%%sql
-- Orders over $20 placed by customer 1.
-- Parentheses are not strictly needed here, but they document intent.
SELECT *
FROM orders
WHERE customer_id = 1
  AND amount > 20;

## 2. Pattern matching, ranges, and set membership

Three idioms cover most of the filtering you will ever write beyond simple comparisons.

- **`LIKE`** for string patterns. `%` matches zero or more characters; `_` matches exactly one. `ILIKE` is the case-insensitive variant (Postgres, DuckDB). MySQL's `LIKE` is case-insensitive by default; SQL Server uses `COLLATE`.
- **`BETWEEN a AND b`** is shorthand for `>= a AND <= b` — **inclusive on both ends**. Most useful for date and amount ranges.
- **`IN (...)`** is shorthand for an `OR` chain of equality checks. `x IN (1, 2, 3)` is the same as `x = 1 OR x = 2 OR x = 3`, but reads better and lets the planner optimize.

In [ ]:
%%sql
-- LIKE: names starting with 'A', case-insensitive emails ending in '.com'.
SELECT name, email
FROM customers
WHERE name LIKE 'A%'
   OR email ILIKE '%.COM';

In [ ]:
%%sql
-- BETWEEN: orders in the first half of April 2024 (inclusive on both ends).
SELECT *
FROM orders
WHERE order_date BETWEEN DATE '2024-04-01' AND DATE '2024-04-14';

In [ ]:
%%sql
-- IN: pick orders from a set of customer ids.
SELECT *
FROM orders
WHERE customer_id IN (1, 3);

## 3. NULL in filters — and the NOT IN trap

Quick recap from notebook 02: `WHERE` treats `UNKNOWN` the same as `FALSE`. Any comparison that touches `NULL` produces `UNKNOWN`, so the row silently disappears. Always use `IS NULL` / `IS NOT NULL` for null checks.

Now the expensive surprise: **`NOT IN` is broken when the list contains `NULL`**. If `NULL` is anywhere inside `NOT IN (...)`, the result is `UNKNOWN` for **every** row — so the query returns **zero rows**, silently.

Why? `x NOT IN (a, b, NULL)` expands to `x <> a AND x <> b AND x <> NULL`. That last comparison is `UNKNOWN`, and `something AND UNKNOWN` is at best `UNKNOWN`, which `WHERE` drops.

Workarounds: filter `NULL`s out of the list explicitly, or use `NOT EXISTS` (notebook 06).

In [ ]:
%%sql
-- The trap: a NULL inside NOT IN swallows all results.
-- We expect Bob and Carol back, but get nothing.
SELECT name
FROM customers
WHERE customer_id NOT IN (1, NULL);

In [ ]:
%%sql
-- The fix: build the list so it cannot contain NULL.
SELECT name
FROM customers
WHERE customer_id NOT IN (
    SELECT customer_id FROM orders WHERE customer_id IS NOT NULL
);

## 4. ORDER BY — direction, multi-column, and NULLs

`ORDER BY` sorts the result set. A few rules worth carrying:

- **Direction is per column.** `ORDER BY a ASC, b DESC` sorts ascending by `a`, then descending by `b` to break ties. `ASC` is the default.
- **You can sort by an expression or a `SELECT` alias.** `SELECT` runs before `ORDER BY` (notebook 02 §5), so aliases are visible.
- **NULL ordering is dialect-defined.** Postgres puts `NULL`s last for `ASC`, first for `DESC`. MySQL puts them first either way. SQL Server puts them first either way. **Use explicit `NULLS FIRST` / `NULLS LAST` when it matters** — standard SQL, supported by Postgres, DuckDB, Oracle, and Snowflake.
- **Avoid ordering by ordinal position** (`ORDER BY 1, 3`). It's terse, but breaks the moment someone reorders the `SELECT` list.

In [ ]:
%%sql
-- Customers sorted by signup_date descending,
-- then by name ascending as a tie-breaker.
-- NULL emails sort last regardless of dialect default.
SELECT name, email, signup_date
FROM customers
ORDER BY signup_date DESC,
         email ASC NULLS LAST,
         name ASC;

## 5. LIMIT, OFFSET, and why deep pagination is slow

`LIMIT n` keeps only the first `n` rows of the result. `LIMIT n OFFSET m` skips `m` rows and then keeps `n`. SQL Server uses `OFFSET m ROWS FETCH NEXT n ROWS ONLY` or the older `TOP n`; the idea is the same.

Two important properties:

- **`LIMIT` without `ORDER BY` is non-deterministic.** "Top 10" only makes sense if you have specified what "top" means.
- **`OFFSET` is linear in the offset value.** To return rows 10,001–10,020, the engine still has to produce, sort, and discard the first 10,000. On large tables, page 500 is dramatically slower than page 1.

**The fix for real pagination is keyset (cursor) pagination**: remember the last value you saw and ask for rows *after* it. Something like `WHERE id > :last_seen_id ORDER BY id LIMIT 20` — constant time per page regardless of how deep you go.

In [ ]:
%%sql
-- Page 2 of orders, 2 rows per page, sorted by amount descending.
SELECT *
FROM orders
ORDER BY amount DESC
LIMIT 2 OFFSET 2;

## 6. Set operations — UNION, INTERSECT, EXCEPT

Set operators **stack the results of two queries vertically** and treat the rows as set members.

- **`UNION`** — combine and deduplicate.
- **`UNION ALL`** — combine without deduplication. Faster, and almost always what you actually want when you know the inputs are already disjoint.
- **`INTERSECT`** — rows that appear in both queries.
- **`EXCEPT`** (called `MINUS` in Oracle) — rows in the left query that are not in the right.

Rules every dialect agrees on:

- Both sides must produce the **same number of columns**.
- Column **types must be compatible** position by position.
- The **result column names come from the first `SELECT`**.
- A single `ORDER BY` / `LIMIT` applies to the **combined** result and goes at the very end.

In [ ]:
%%sql
-- A small prospects table for set-operation demos.
-- Prospects are people on the newsletter who haven't bought yet.
-- Note: Alice appears in both tables (she signed up, then bought).
DROP TABLE IF EXISTS prospects;
CREATE TABLE prospects (
    prospect_id  INTEGER PRIMARY KEY,
    name         VARCHAR NOT NULL,
    email        VARCHAR,
    signup_date  DATE    NOT NULL
);
INSERT INTO prospects VALUES
    (1, 'Dave',  'dave@example.com',  DATE '2024-05-01'),
    (2, 'Erin',  'erin@example.com',  DATE '2024-05-15'),
    (3, 'Alice', 'alice@example.com', DATE '2024-06-01');
SELECT * FROM prospects;

In [ ]:
%%sql
-- UNION: every contactable person, with duplicates removed.
-- Alice appears in both tables but shows up once.
SELECT name, email FROM customers
UNION
SELECT name, email FROM prospects
ORDER BY name;

In [ ]:
%%sql
-- UNION ALL: faster, keeps duplicates.
-- Alice now appears twice — once from each source.
SELECT name, email FROM customers
UNION ALL
SELECT name, email FROM prospects
ORDER BY name;

In [ ]:
%%sql
-- INTERSECT: people who are in both lists (prospects who became customers).
SELECT name, email FROM customers
INTERSECT
SELECT name, email FROM prospects;

In [ ]:
%%sql
-- EXCEPT: prospects who have NOT yet become customers.
SELECT name, email FROM prospects
EXCEPT
SELECT name, email FROM customers;

## 7. Common pitfalls (carry forward)

- **`NOT IN` with `NULL` in the list returns zero rows.** Either guarantee no `NULL`s in the list (filter with `IS NOT NULL`) or rewrite with `NOT EXISTS`.
- **`LIKE` with a leading `%`** (`'%foo%'`) cannot use a standard B-tree index — it forces a full scan. Use trigram indexes or full-text search for that pattern.
- **`BETWEEN` is inclusive on both ends.** For half-open ranges over timestamps ("all of April"), prefer `>= '2024-04-01' AND < '2024-05-01'` to avoid the last-millisecond-of-the-month bug.
- **NULL ordering defaults differ across dialects.** Write `NULLS FIRST` / `NULLS LAST` explicitly whenever it could affect correctness.
- **`LIMIT` without `ORDER BY`** gives non-deterministic results. Add the order, even in throwaway queries — habits stick.
- **Reaching for `UNION` when `UNION ALL` is correct.** `UNION` does an implicit dedup that costs a sort or a hash table; only use it when you actually expect duplicates and want them collapsed.
- **Mixing `AND` and `OR` without parentheses.** `AND` binds tighter — `a OR b AND c` is `a OR (b AND c)`, which is rarely what you mean.

Next up: **notebook 04 — Joins**, where we connect `customers`, `orders`, and `prospects` and meet the four standard join types, plus the surprises hiding in outer joins and `NULL`.